# Ablation 4.3.6 — Comparison with the generative pipeline (Campi et al., CVPRW 2025)

**Thesis:** *Concept dataset extraction for Concept Activation Vectors: A study on open-vocabulary grounding and segmentation*  
**Author:** Alessandro Cogollo (233022)  
**Advisor:** Prof. Marco Brambilla — **Co-advisors:** A. De Santis, M. Bianchi, R. Campi  
**Academic year:** 2025-2026

**Maps to:** Thesis §4.3.6 / Executive Summary RQ4 — *Comparison with generated concepts*.

## Purpose
Re-implements the evaluation protocol of *Towards Synthetic Concept Activation Vectors via Generative Models* and applies it to the **extracted** concept dataset of this thesis, so the `extracted` row can be read side-by-side with the paper's Table 2 (generated ↔ real cosine).

## Inputs (Kaggle datasets)
- Real reference (positives): ZeroShot-T2I bare subfolder `concepts/{concept}/{concept}`, fallback DTD `{concept}`.
- Extracted set: `acg4cav-generated-concepts/.../crops/{concept}`.
- Negatives: 500 random ImageNet-1k images from `vtcav-dataset/random` (shared across all CAVs).

## Outputs (`/kaggle/working`)
`cav_similarity_raw.csv`, `cav_similarity_normalized.csv`, `table2_by_source.csv`, `table2_by_type.csv`, `table3_intra_similarity.csv`, `figure5_intra_vs_size.png`, `summary.json`.

## How to run
Run top-to-bottom on Kaggle with the four datasets above attached. Set `MAX_CONCEPTS`/`MAX_POS` for a quick smoke run; leave `None` for the full 34-concept comparison.

## ⚠️ Notes for reviewers (read before interpreting the numbers)
1. **Backbone mismatch — by design of this protocol.** This notebook uses the **Keras / ImageNet** backbones of Campi et al. (`ResNet50V2`, `InceptionV3`, `VGG16`, last three conv blocks). The rest of the thesis uses **torchvision `ResNet50` (IMAGENET1K_V1)**. Cosine values here are therefore **comparable to Campi's Table 2, not directly to the thesis' §4.2 numbers**. This is intentional: the protocol is fixed to the SOTA paper to make the `extracted` row readable next to theirs.
2. **Unfiltered vs filtered.** Campi et al. apply a CLIP filter after generation; the proposed extraction pipeline does **not**. The comparison is thus *unfiltered extracted* vs *filtered generated*; the resulting gap (~0.10) is a **lower bound** for the extraction approach.
3. **Layers are fixed, not heuristic.** The per-backbone layers in `CNN_CONFIG` are the last three convolutional blocks and are kept fixed for coherence with the Visual-TCAV setup.

# ACG4CAV — Riproduzione dell'esperimento di similarità CAV (Campi et al., CVPRW 2025)

Questo notebook **replica le scelte sperimentali** del paper *"Towards Synthetic Concept Activation
Vectors via Generative Models"* e le applica per **confrontare il dataset di concetti che hai estratto tu**
(`extracted`) con la **ground-truth reale**, nello stesso identico framework di valutazione del paper.

> La baseline generativa (ZeroShot T2I) **non** viene calcolata: i suoi valori di riferimento sono già
> nella Table 2 del paper (riportata sotto), e ti basta leggere la riga `extracted` accanto a quelli.

**Scelte sperimentali replicate dal paper (Sez. 4.3–4.4):**
- CAV via **Difference of Means** (centroide positivo − centroide negativo), *non* classificatori lineari [17].
- 3 CNN pre-addestrate su **ImageNet-1k** (Keras): **ResNet50V2, InceptionV3, VGG16**.
- **Ultimi 3 layer convoluzionali** per ogni CNN, calcolati separatamente.
- **500 immagini random da ImageNet-1k** come negativi (qui: cartella `random` del VTCAV Dataset).
- Due strategie di calcolo del centroide: **GAP** (Visual-TCAV [5]) e **flatten** (originale [13]).
- Metrica: **cosine similarity** tra CAV estratto e CAV reale dello stesso concetto.
- **Intra-similarity** della ground-truth (split in 2 sottogruppi, CAV per gruppo, cosine, 25 run → 450 misure/concetto).
- Aggregazione su: concetti, CNN, layer, strategia di centroide; raggruppata per fonte e tipo.

**Mappatura dei tuoi dataset:**

| Ruolo nel paper | Tuo dataset | Path (in ordine di priorità) |
|---|---|---|
| **Riferimento manuale/reale** (positivi) | ZeroShot T2I → fallback DTD | `…/zeroshot-t2i-concepts/concepts/{concept}/{concept}` → `…/describable-textures-dataset-dtd/{concept}` |
| **Negativi** (500 da ImageNet-1k) | VTCAV Dataset → `random` | `…/vtcav-dataset/random/*.JPEG` |
| **Dataset valutato → ESTRATTO (tuo)** | ACG4CAV – Generated Concepts | `…/acg4cav-generated-concepts/generated concepts/crops/{concept}` |


## 0. Configurazione

I path sono quelli che hai indicato (prefisso `/kaggle/input/datasets/<owner>/<slug>/`).
La cella stampa OK/NO per ogni cartella e, se qualcosa manca, elenca cosa c'è sotto
`/kaggle/input/datasets/` così correggi subito lo slug.

In [1]:
import os, gc, glob, json, math, random, warnings, re
from collections import defaultdict
import numpy as np

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
from tensorflow import keras
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

# ============================================================================
# === PATH DEI DATASET (riferimenti forniti) =================================
# ============================================================================
# Riferimento "manuale/reale" (positivi), in ordine di PRIORITÀ:
#   1) ZeroShot T2I: .../concepts/{concept}/{concept}   (sottocartella nuda, non _flux/_gpt/_sd)
#   2) DTD:          .../describable-textures-dataset-dtd/{concept}
T2I_CONCEPTS_DIR = "/kaggle/input/datasets/alessandrocogollo/zeroshot-t2i-concepts/concepts"
DTD_DIRS = [
    "/kaggle/input/datasets/jmexpert/describable-textures-dataset-dtd",
    "/kaggle/input/datasets/jmexpert/describable-textures-dataset-dtd/dtd/images",  # layout DTD classico
]
# Il TUO dataset estratto: .../generated concepts/crops/{concept}
EXTRACTED_DIR = "/kaggle/input/datasets/alessandrocogollo/acg4cav-generated-concepts/generated concepts/crops"
# Negativi (500 random ImageNet-1k): .../vtcav-dataset/random
NEG_DIR = "/kaggle/input/datasets/alessandrocogollo/vtcav-dataset/random"

OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================================
# === PARAMETRI SPERIMENTALI (come nel paper) ================================
# ============================================================================
N_NEGATIVES        = 500                  # negativi random da ImageNet-1k (Sez. 4.3)
CENTROID_STRATS    = ["gap", "flatten"]   # GAP [5] + flatten [13]
INTRA_RUNS         = 25                   # 25 run → 450 misure/concetto (Sez. 4.4)
INTRA_SUBGROUP_SIZES = [10, 20, 30, 40, 50, None]  # None = metà del dataset (Figure 5)
BATCH              = 32
CHUNK              = 64                    # immagini caricate in RAM per volta (gestione memoria)

# Un'unica variante da valutare: il TUO dataset estratto vs il riferimento.
VARIANTS_TO_EVAL   = ["extracted"]

# Limiti per run veloci (imposta MAX_CONCEPTS / MAX_POS_IMAGES per uno smoke test, None = tutti)
MAX_CONCEPTS   = None
MAX_POS_IMAGES = None
IMG_EXTS = (".jpg", ".jpeg", ".png", ".JPEG", ".JPG", ".PNG", ".bmp", ".webp")

# --- Diagnostica path: avvisa subito se gli slug sono sbagliati ---
def _check_paths():
    checks = [("T2I_CONCEPTS_DIR", T2I_CONCEPTS_DIR), ("EXTRACTED_DIR", EXTRACTED_DIR), ("NEG_DIR", NEG_DIR)]
    checks += [(f"DTD_DIRS[{i}]", d) for i, d in enumerate(DTD_DIRS)]
    for label, p in checks:
        print(f"  {'OK ' if os.path.isdir(p) else 'NO '} {label}: {p}")
    root = "/kaggle/input/datasets"
    if os.path.isdir(root):
        for owner in sorted(os.listdir(root)):
            ob = os.path.join(root, owner)
            if os.path.isdir(ob):
                print(f"  /kaggle/input/datasets/{owner}/ -> {os.listdir(ob)}")
_check_paths()


E0000 00:00:1782749603.371088      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782749603.443308      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782749604.081514      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782749604.081568      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782749604.081571      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782749604.081574      16 computation_placer.cc:177] computation placer already registered. Please check linka

TF: 2.19.0 | GPU: []
  NO  T2I_CONCEPTS_DIR: /kaggle/input/datasets/alessandrocogollo/zeroshot-t2i-concepts/concepts
  NO  EXTRACTED_DIR: /kaggle/input/datasets/alessandrocogollo/acg4cav-generated-concepts/generated concepts/crops
  NO  NEG_DIR: /kaggle/input/datasets/alessandrocogollo/vtcav-dataset/random
  NO  DTD_DIRS[0]: /kaggle/input/datasets/jmexpert/describable-textures-dataset-dtd
  NO  DTD_DIRS[1]: /kaggle/input/datasets/jmexpert/describable-textures-dataset-dtd/dtd/images


## 1. Metadati dei 34 concetti (Table 1 del paper)

Servono solo per **aggregare** i risultati per *fonte* (Search engine / ImageNet / DTD) e per *tipo*
(Object O / Texture T), come la Table 2. Il caricamento delle immagini avviene per cartella,
quindi se hai estratto solo un sottoinsieme dei concetti il notebook usa l'intersezione disponibile.

In [2]:
# concept_key : (source, type) ; source in {search, imagenet, dtd}; type in {O, T}
CONCEPTS = {
    # --- Search engines (10) ---
    "cast_iron":("search","O"), "fin":("search","O"), "glass":("search","O"),
    "grass":("search","T"), "leopard_printed":("search","T"), "newspaper":("search","O"),
    "pen":("search","O"), "sphere":("search","O"), "spotted":("search","T"), "taxi_sign":("search","O"),
    # --- ImageNet (10) ---
    "asparagus":("imagenet","O"), "cloister":("imagenet","O"), "feline":("imagenet","O"),
    "grandfather":("imagenet","O"), "guitarist":("imagenet","O"), "kitchen":("imagenet","O"),
    "lichen":("imagenet","O"), "rodent":("imagenet","O"), "steeple":("imagenet","O"), "tattoo":("imagenet","O"),
    # --- DTD (14) ---
    "bubbly":("dtd","T"), "chequered":("dtd","T"), "cracked":("dtd","T"), "crystalline":("dtd","T"),
    "dotted":("dtd","T"), "frilly":("dtd","T"), "honeycombed":("dtd","T"), "meshed":("dtd","T"),
    "perforated":("dtd","T"), "spiralled":("dtd","T"), "striped":("dtd","T"), "waffled":("dtd","T"),
    "woven":("dtd","T"), "wrinkled":("dtd","T"),
}

# Alias per tollerare piccole differenze nei nomi cartella
ALIASES = {
    "leopard_printed": ["leopard_print","leopard_pattern","leopard","leopard_p","leopard printed"],
    "honeycombed":     ["honeycomb","honey_combed"],
    "taxi_sign":       ["taxi","taxisign","taxi sign"],
    "cast_iron":       ["castiron","cast iron"],
}

def normalize(name:str)->str:
    return re.sub(r"[^a-z0-9]+","_", name.strip().lower()).strip("_")

def resolve_key(folder_name:str):
    n = normalize(folder_name)
    if n in CONCEPTS: return n
    for k, al in ALIASES.items():
        if n == k or n in [normalize(a) for a in al]: return k
    return None

print(f"{len(CONCEPTS)} concetti definiti "
      f"(search={sum(s=='search' for s,_ in CONCEPTS.values())}, "
      f"imagenet={sum(s=='imagenet' for s,_ in CONCEPTS.values())}, "
      f"dtd={sum(s=='dtd' for s,_ in CONCEPTS.values())}).")


34 concetti definiti (search=10, imagenet=10, dtd=14).


## 2. CNN and layers (Sez. 4.3)

Three ImageNet-pretrained Keras CNNs, **last three convolutional layers** each. These layers are **fixed** to match the Campi et al. / Visual-TCAV evaluation and must not be silently changed — any change breaks comparability with the paper's Table 2.

In [3]:
CNN_CONFIG = {
    "ResNet50V2": dict(
        builder=keras.applications.ResNet50V2,
        preprocess=keras.applications.resnet_v2.preprocess_input,
        size=224,
        layers=["conv5_block1_out", "conv5_block2_out", "conv5_block3_out"],
    ),
    "InceptionV3": dict(
        builder=keras.applications.InceptionV3,
        preprocess=keras.applications.inception_v3.preprocess_input,
        size=299,
        layers=["mixed8", "mixed9", "mixed10"],
    ),
    "VGG16": dict(
        builder=keras.applications.VGG16,
        preprocess=keras.applications.vgg16.preprocess_input,
        size=224,
        layers=["block5_conv1", "block5_conv2", "block5_conv3"],
    ),
}

_model_cache = {}
def get_feature_model(name):
    if name in _model_cache:
        return _model_cache[name]
    cfg = CNN_CONFIG[name]
    base = cfg["builder"](weights="imagenet", include_top=False)
    outs = [base.get_layer(l).output for l in cfg["layers"]]
    m = keras.Model(inputs=base.input, outputs=outs, name=f"{name}_features")
    _model_cache[name] = m
    return m


## 3. Utility: caricamento immagini, estrazione attivazioni, pooling, CAV

In [4]:
def list_images(folder, max_n=None):
    if not folder or not os.path.isdir(folder):
        return []
    files = sorted([os.path.join(folder, f) for f in os.listdir(folder)
                    if f.endswith(IMG_EXTS)])
    if max_n is not None:
        files = files[:max_n]
    return files

def load_batch(paths, size, preprocess):
    arrs = []
    for p in paths:
        try:
            img = keras.utils.load_img(p, target_size=(size, size))
            arrs.append(keras.utils.img_to_array(img))
        except Exception as e:
            print("  [skip]", os.path.basename(p), "->", e)
    if not arrs:
        return None
    return preprocess(np.stack(arrs, 0).astype("float32"))

def pool(acts, strategy):
    # acts: (N, H, W, C)
    if strategy == "gap":
        return acts.mean(axis=(1, 2))                 # (N, C)
    if strategy == "flatten":
        return acts.reshape(acts.shape[0], -1)        # (N, H*W*C)
    raise ValueError(strategy)

def centroid_chunked(model, paths, size, preprocess, strategies):
    '''Centroide (media) per ogni (layer_idx, strategia) calcolato in streaming -> memory-safe.'''
    sums, count = {}, 0
    for i in range(0, len(paths), CHUNK):
        x = load_batch(paths[i:i+CHUNK], size, preprocess)
        if x is None:
            continue
        acts = model.predict(x, batch_size=BATCH, verbose=0)
        if not isinstance(acts, list):
            acts = [acts]
        for li, a in enumerate(acts):
            for s in strategies:
                v = pool(a, s).sum(axis=0)
                sums[(li, s)] = v if (li, s) not in sums else sums[(li, s)] + v
        count += x.shape[0]
        del x, acts; gc.collect()
    if count == 0:
        return None, 0
    return {k: v / count for k, v in sums.items()}, count

def per_image_pooled(model, paths, size, preprocess, strategies):
    '''Ritorna dict[(layer_idx, strategy)] -> (N, D). Usato per l'intra-similarity (split).'''
    store = defaultdict(list)
    for i in range(0, len(paths), CHUNK):
        x = load_batch(paths[i:i+CHUNK], size, preprocess)
        if x is None:
            continue
        acts = model.predict(x, batch_size=BATCH, verbose=0)
        if not isinstance(acts, list):
            acts = [acts]
        for li, a in enumerate(acts):
            for s in strategies:
                store[(li, s)].append(pool(a, s))
        del x, acts; gc.collect()
    return {k: np.concatenate(v, 0) for k, v in store.items()}

def cosine(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


## 4. Risoluzione delle cartelle e diagnostica copertura

- **riferimento manuale/reale** (`real_folder`): priorità a ZeroShot T2I `concepts/{concept}/{concept}`
  (la sottocartella *nuda*, non `_flux_1_1`/`_gpt_i_1`/`_sd_3_5`); se assente, fallback su DTD `{concept}`.
- **extracted**: `…/generated concepts/crops/{concept}`

`match_subdir` confronta i nomi cartella tollerando alias e maiuscole/underscore.

In [5]:
def match_subdir(base, key):
    '''Sottocartella di `base` che corrisponde al concetto (per nome/alias), o None.'''
    if not os.path.isdir(base):
        return None
    targets = {normalize(c) for c in [key] + ALIASES.get(key, [])}
    for d in sorted(os.listdir(base)):
        full = os.path.join(base, d)
        if os.path.isdir(full) and (normalize(d) in targets or resolve_key(d) == key):
            return full
    return None

def real_folder(key):
    '''Riferimento manuale/reale. Priorità: T2I concepts/{c}/{c}; fallback DTD/{c}.'''
    # 1) ZeroShot T2I: concepts/{concept}/{concept} (sottocartella nuda, non _flux/_gpt/_sd)
    c = match_subdir(T2I_CONCEPTS_DIR, key)
    if c:
        inner = match_subdir(c, key)
        if inner and list_images(inner, 1):
            return inner
        if list_images(c, 1):           # fallback: immagini direttamente in concepts/{concept}
            return c
    # 2) DTD: {concept}
    for base in DTD_DIRS:
        d = match_subdir(base, key)
        if d and list_images(d, 1):
            return d
    return None

def extracted_folder(key):
    '''Il tuo dataset estratto: crops/{concept}.'''
    e = match_subdir(EXTRACTED_DIR, key)
    if e and list_images(e, 1):
        return e
    return None

def variant_folder(variant, key):
    if variant == "extracted":
        return extracted_folder(key)
    raise ValueError(f"Variante non supportata: {variant}")

# --- Diagnostica copertura ---
discovered = []
for key in CONCEPTS:
    rf, ef = real_folder(key), extracted_folder(key)
    discovered.append(dict(concept=key, source=CONCEPTS[key][0], type=CONCEPTS[key][1],
                           real=len(list_images(rf)) if rf else 0,
                           extracted=len(list_images(ef)) if ef else 0,
                           real_path=rf or "", extracted_path=ef or ""))

cov = pd.DataFrame(discovered).set_index("concept")
print("Copertura immagini per concetto:")
display(cov[["source", "type", "real", "extracted"]])

USABLE = [k for k in CONCEPTS if cov.loc[k, "real"] > 0 and cov.loc[k, "extracted"] > 0]
if MAX_CONCEPTS:
    USABLE = USABLE[:MAX_CONCEPTS]
print(f"\nConcetti utilizzabili (riferimento + estratto entrambi presenti): {len(USABLE)}")
print(USABLE)

missing_real = [k for k in CONCEPTS if cov.loc[k, "real"] == 0 and cov.loc[k, "extracted"] > 0]
if missing_real:
    print(f"\n⚠️ Estratti ma SENZA riferimento manuale/reale trovato (controlla nomi cartella): {missing_real}")
    display(cov.loc[missing_real, ["extracted_path"]])


Copertura immagini per concetto:


,source,type,real,extracted
concept,,,,
cast_iron,search,O,0,0
fin,search,O,0,0
glass,search,O,0,0
grass,search,T,0,0
leopard_printed,search,T,0,0
newspaper,search,O,0,0
pen,search,O,0,0
sphere,search,O,0,0
spotted,search,T,0,0



Concetti utilizzabili (riferimento + estratto entrambi presenti): 0
[]


## 5. Centroidi negativi (500 ImageNet-1k, condivisi)

Gli **stessi** 500 negativi sono usati per *tutti* i CAV (reali ed estratti), come nel paper.
Sono condivisi: questo è ciò che rende confrontabili le cosine similarity (il termine negativo
non si "cancella" nella cosine, quindi deve essere identico ovunque).

In [6]:
neg_paths = list_images(NEG_DIR, N_NEGATIVES)
assert len(neg_paths) > 0, f"Nessun negativo trovato in {NEG_DIR} — controlla il path."
print(f"Negativi: {len(neg_paths)} immagini da {NEG_DIR}")

NEG_CENTROIDS = {}   # NEG_CENTROIDS[cnn][(layer_idx, strat)] -> vector
for cnn, cfg in CNN_CONFIG.items():
    print(f"  Estrazione centroidi negativi — {cnn} ...")
    m = get_feature_model(cnn)
    cents, n = centroid_chunked(m, neg_paths, cfg["size"], cfg["preprocess"], CENTROID_STRATS)
    NEG_CENTROIDS[cnn] = cents
    print(f"    ok ({n} immagini)")


AssertionError: Nessun negativo trovato in /kaggle/input/datasets/alessandrocogollo/vtcav-dataset/random — controlla il path.

## 6. Calcolo dei CAV e valutazione (cosine similarity extracted ↔ reale)

Per ogni `(concetto, CNN, layer, strategia)`:
1. centroide positivo dalle immagini **estratte**,
2. `CAV_extracted = centroide_positivo − centroide_negativo` (Difference of Means),
3. lo stesso dalle immagini **reali** → `CAV_real`,
4. `cosine(CAV_extracted, CAV_real)`.

In [ ]:
def compute_cavs_for_paths(cnn, paths, size, preprocess):
    '''Ritorna dict[(layer_idx, strat)] -> CAV (pos_centroid - neg_centroid).'''
    pos, n = centroid_chunked(get_feature_model(cnn), paths, size, preprocess, CENTROID_STRATS)
    if pos is None:
        return None
    negc = NEG_CENTROIDS[cnn]
    return {k: pos[k] - negc[k] for k in pos}

results = []          # una riga per (concept, cnn, layer, strat)
real_cav_cache = {}   # (concept, cnn) -> cavs reali

for ci, key in enumerate(USABLE, 1):
    src, typ = CONCEPTS[key]
    print(f"[{ci}/{len(USABLE)}] {key} ({src}/{typ})")
    for cnn, cfg in CNN_CONFIG.items():
        size, prep, layers = cfg["size"], cfg["preprocess"], cfg["layers"]

        # CAV reali (cache)
        if (key, cnn) not in real_cav_cache:
            rpaths = list_images(real_folder(key), MAX_POS_IMAGES)
            real_cav_cache[(key, cnn)] = compute_cavs_for_paths(cnn, rpaths, size, prep) if rpaths else None
        real_cavs = real_cav_cache[(key, cnn)]
        if real_cavs is None:
            continue

        # CAV estratti
        epaths = list_images(extracted_folder(key), MAX_POS_IMAGES)
        if not epaths:
            continue
        ext_cavs = compute_cavs_for_paths(cnn, epaths, size, prep)
        if ext_cavs is None:
            continue

        for li, lname in enumerate(layers):
            for s in CENTROID_STRATS:
                sim = cosine(ext_cavs[(li, s)], real_cavs[(li, s)])
                results.append(dict(variant="extracted", concept=key, source=src, type=typ,
                                    cnn=cnn, layer=lname, strategy=s, cosine=sim,
                                    n_pos=len(epaths)))
    gc.collect()

res_df = pd.DataFrame(results)
res_df.to_csv(os.path.join(OUT_DIR, "cav_similarity_raw.csv"), index=False)
print(f"\nTotale misure: {len(res_df)}")
res_df.head()


## 7. Aggregazione in stile *Table 2*

Riga = `extracted` (il tuo dataset), colonne = fonte e tipo.
Valore = `media ± deviazione standard` della cosine similarity, mediata su concetti, CNN, layer e strategie.

In [ ]:
def agg_pm(df, group_col, val="cosine"):
    g = df.groupby([group_col])[val].agg(["mean", "std"]).reset_index()
    g["cell"] = g.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}", axis=1)
    return g.set_index(group_col)["cell"].to_frame("extracted").T

print("=== Aggregato per FONTE (Search eng. / ImageNet / DTD) ===")
display(agg_pm(res_df, "source"))

print("\n=== Aggregato per TIPO (Objects / Textures) + ALL ===")
by_type = agg_pm(res_df, "type")
ov = res_df["cosine"].agg(["mean", "std"])
by_type["All"] = f"{ov['mean']:.2f} ± {ov['std']:.2f}"
display(by_type)

# salva versioni numeriche
res_df.groupby("source")["cosine"].agg(["mean","std"]).to_csv(os.path.join(OUT_DIR,"table2_by_source.csv"))
res_df.groupby("type")["cosine"].agg(["mean","std"]).to_csv(os.path.join(OUT_DIR,"table2_by_type.csv"))


### 7b. Confronto diretto con i numeri del paper (Table 2)

Valori riportati da Campi et al. (cosine similarity *generato ↔ reale*).
**Leggi la tua riga `extracted` della cella precedente accanto a queste righe.**

In [ ]:
paper_table2 = pd.DataFrame({
    "Search eng.": ["0.47 ± 0.19","0.66 ± 0.18","0.52 ± 0.17","0.57 ± 0.15"],
    "ImageNet":    ["0.47 ± 0.12","0.78 ± 0.08","0.71 ± 0.12","0.70 ± 0.11"],
    "DTD":         ["0.73 ± 0.14","0.81 ± 0.07","0.71 ± 0.15","0.72 ± 0.12"],
    "Objects (O)": ["0.62 ± 0.21","0.73 ± 0.15","0.62 ± 0.17","0.64 ± 0.15"],
    "Textures (T)":["0.73 ± 0.14","0.81 ± 0.07","0.71 ± 0.15","0.72 ± 0.12"],
    "All":         ["0.67 ± 0.19","0.77 ± 0.12","0.66 ± 0.17","0.68 ± 0.14"],
}, index=["SD 1.5","SD 1.5 + DB","SD XL 1.0","SD XL Turbo"])
print("Paper (Campi et al., Table 2) — riferimento:")
display(paper_table2)


## 8. Intra-similarity della ground-truth (Table 3 + Figure 5)

Per ogni concetto reale: split casuale in 2 sottogruppi → CAV per sottogruppo → cosine tra i due.
Ripetuto su `(CNN × layer × strategia) = 18` combinazioni e su `INTRA_RUNS=25` run → 450 misure/concetto.
È il **tetto** della metrica: indica quanta similarità si perde per **solo** errore di stima.

In [ ]:
def intra_similarity_concept(key, subgroup_size=None, runs=INTRA_RUNS):
    """Ritorna lista di cosine (una per CNN x layer x strategia x run)."""
    rpaths = list_images(real_folder(key), MAX_POS_IMAGES)
    if len(rpaths) < 4:
        return []
    measures = []
    for cnn, cfg in CNN_CONFIG.items():
        pooled = per_image_pooled(get_feature_model(cnn), rpaths,
                                  cfg["size"], cfg["preprocess"], CENTROID_STRATS)
        if not pooled:                       # nessuna immagine caricata con successo
            gc.collect(); continue
        negc = NEG_CENTROIDS[cnn]
        # N = numero REALE di immagini caricate (load_batch puo' saltarne alcune
        # non decodificabili), NON len(rpaths): altrimenti la permutazione genera
        # indici fuori dai limiti di `pooled`.
        N = next(iter(pooled.values())).shape[0]
        half = min(subgroup_size if subgroup_size else N // 2, N // 2)
        if half < 2:
            del pooled; gc.collect(); continue
        for _ in range(runs):
            idx = np.random.permutation(N)
            g1, g2 = idx[:half], idx[half:2*half]
            for li, lname in enumerate(cfg["layers"]):
                for s in CENTROID_STRATS:
                    P = pooled[(li, s)]
                    cav1 = P[g1].mean(0) - negc[(li, s)]
                    cav2 = P[g2].mean(0) - negc[(li, s)]
                    measures.append(cosine(cav1, cav2))
        del pooled; gc.collect()
    return measures

intra_rows = []
real_concepts = [k for k in USABLE if cov.loc[k, "real"] >= 4]
print(f"Intra-similarity su {len(real_concepts)} concetti (subgroup = meta' del dataset)...")
for ci, key in enumerate(real_concepts, 1):
    m = intra_similarity_concept(key, subgroup_size=None, runs=INTRA_RUNS)
    if m:
        intra_rows.append(dict(concept=key, source=CONCEPTS[key][0], type=CONCEPTS[key][1],
                               intra_mean=float(np.mean(m)), intra_std=float(np.std(m)), n=len(m)))
    print(f"  [{ci}/{len(real_concepts)}] {key}: {np.mean(m):.3f}" if m else f"  [{ci}] {key}: n/a")

intra_df = pd.DataFrame(intra_rows)
intra_df.to_csv(os.path.join(OUT_DIR, "table3_intra_similarity.csv"), index=False)
print("\nIntra-similarity media per fonte:")
display(intra_df.groupby("source")["intra_mean"].mean().round(3).to_frame("mean"))
display(intra_df.set_index("concept")[["source","type","intra_mean","intra_std"]])

### 8b. Figure 5 — intra-similarity vs dimensione del sottogruppo

In [ ]:
import matplotlib.pyplot as plt

curve = []
sample_concepts = real_concepts[:min(len(real_concepts), 12)]  # sottoinsieme per velocità
for size in INTRA_SUBGROUP_SIZES:
    vals = []
    for key in sample_concepts:
        m = intra_similarity_concept(key, subgroup_size=size, runs=max(5, INTRA_RUNS//3))
        if m:
            vals.extend(m)
    if vals:
        label = size if size else "~half"
        curve.append((label, float(np.mean(vals)), float(np.std(vals))))
        print(f"  subgroup={label}: {np.mean(vals):.3f} ± {np.std(vals):.3f}")

if curve:
    xs = list(range(len(curve)))
    means = np.array([c[1] for c in curve]); stds = np.array([c[2] for c in curve])
    plt.figure(figsize=(7,4))
    plt.plot(xs, means, "-o", color="red", label="Mean")
    plt.fill_between(xs, means-stds, means+stds, color="gray", alpha=0.4, label="Std dev")
    plt.xticks(xs, [str(c[0]) for c in curve])
    plt.xlabel("# samples per subgroup"); plt.ylabel("Cosine similarity")
    plt.title("Intra-similarity vs subgroup dimensions")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "figure5_intra_vs_size.png"), dpi=130)
    plt.show()


### 8c. (Opzionale) Cosine normalizzata sul tetto di intra-similarity

`cosine_norm = cosine / intra_mean` (concept-wise): ~1.0 significa "indistinguibile dal reale
entro l'errore di stima". Tieni questa **accanto** ai valori grezzi, non al posto loro — i grezzi
sono quelli confrontabili con la Table 2 del paper. Nota: essendo il tetto empirico e calcolato su
metà campione, qualche valore può occasionalmente superare 1.

In [ ]:
if len(res_df) and len(intra_df):
    norm = res_df.merge(intra_df[["concept", "intra_mean"]], on="concept", how="left")
    norm["cosine_norm"] = norm["cosine"] / norm["intra_mean"]
    norm.to_csv(os.path.join(OUT_DIR, "cav_similarity_normalized.csv"), index=False)

    nn = norm.dropna(subset=["cosine_norm"])
    print("=== Cosine NORMALIZZATA per fonte (cosine / intra_mean) ===")
    display(agg_pm(nn, "source", val="cosine_norm"))
    bt = agg_pm(nn, "type", val="cosine_norm")
    ovn = nn["cosine_norm"].agg(["mean","std"])
    bt["All"] = f"{ovn['mean']:.2f} ± {ovn['std']:.2f}"
    display(bt)
else:
    print("res_df o intra_df vuoti: salto la normalizzazione.")


## 9. Riepilogo e salvataggio

Output in `/kaggle/working`:
- `cav_similarity_raw.csv` — ogni misura `(concept, cnn, layer, strategy)` (grezza)
- `cav_similarity_normalized.csv` — con `cosine_norm`
- `table2_by_source.csv`, `table2_by_type.csv` — aggregati stile Table 2
- `table3_intra_similarity.csv` — intra-similarity per concetto (Table 3)
- `figure5_intra_vs_size.png` — Figure 5
- `summary.json` — sintesi finale

In [ ]:
summary = {
    "n_concepts_used": len(USABLE),
    "concepts_used": USABLE,
    "cnns": list(CNN_CONFIG.keys()),
    "centroid_strategies": CENTROID_STRATS,
    "n_negatives": len(neg_paths),
    "extracted_vs_real_cosine": res_df["cosine"].agg(["mean","std"]).round(4).to_dict() if len(res_df) else {},
    "extracted_vs_real_by_source": res_df.groupby("source")["cosine"].mean().round(4).to_dict() if len(res_df) else {},
    "extracted_vs_real_by_type": res_df.groupby("type")["cosine"].mean().round(4).to_dict() if len(res_df) else {},
    "intra_similarity_overall": float(intra_df["intra_mean"].mean()) if len(intra_df) else None,
}
with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print("\nFatto. File in /kaggle/working.")
